# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/3bud-ZC/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Before writing a rule, inspect the shape of the core signals. Visibility, clicks, and sessions are heavy-tailed; ratios and position have very different scales. The audit uses medians/quantiles rather than assuming normally distributed features.

In [1]:
import os, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

REPO_URL="https://github.com/3bud-ZC/flyrank-ml-internship"
REPO_DIR="flyrank-ml-internship"

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"data/raw/content_refresh_anonymized.csv").exists(): return p
    return None

root=find_root()
if root is None:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR],check=True)
    root=Path(REPO_DIR).resolve()
os.chdir(root)

df=pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["target"]=df["trend_direction"].str.lower().eq("down").astype(int)
fields=["impressions_90d","clicks_90d","sessions_90d","avg_position","ctr","content_age_days","days_since_last_update"]
print(df[fields].quantile([0,.25,.5,.75,.9,.99,1]).round(2).to_string())


      impressions_90d  clicks_90d  sessions_90d  avg_position     ctr  content_age_days  days_since_last_update
0.00             1.00        0.00          1.00           0.0    0.00              90.0                     1.0
0.25            81.00        0.00          2.00           6.2    0.00             132.0                    20.0
0.50           731.00        1.00          7.00          10.8    0.07             236.0                    20.0
0.75          3615.25        7.00         27.00          22.3    0.29             333.0                   104.0
0.90         12136.40       32.00         88.00          36.8    0.65             463.0                   104.0
0.99         73505.83      253.01        451.01          69.9    8.33             537.0                   106.0
1.00        517715.00     4178.00       4345.00         245.0  100.00             564.0                   373.0


## 2. Signal test #1 / #2 / #3 (verdict each)

Three safe, feature-time signals are checked against the starter proxy:

1. **Content age** — do decline and non-decline groups differ directionally?
2. **Visibility** — does a simple high-impression bucket have a different proxy rate?
3. **CTR opportunity** — among visible pages in positions 1–20, does lower CTR correspond to a different proxy rate?

These are descriptive checks, not causal tests.

In [2]:
tests={}

age=df.groupby("target")["content_age_days"].median()
tests["age"]={
    "negative_median":float(age.get(0,np.nan)),
    "positive_median":float(age.get(1,np.nan)),
    "verdict":"MIXED" if abs(age.get(1,0)-age.get(0,0))<30 else "DIRECTIONAL"
}

high_vis=df["impressions_90d"]>=df["impressions_90d"].median()
tests["visibility"]={
    "high_visibility_proxy_rate":float(df.loc[high_vis,"target"].mean()),
    "low_visibility_proxy_rate":float(df.loc[~high_vis,"target"].mean()),
    "verdict":"MIXED"
}

eligible=(df["impressions_90d"]>=500)&(df["avg_position"].between(1,20))
low_ctr=eligible&(df["ctr"]<0.5)
normal_ctr=eligible&(df["ctr"]>=0.5)
tests["low_ctr_visible"]={
    "low_ctr_proxy_rate":float(df.loc[low_ctr,"target"].mean()) if low_ctr.any() else None,
    "other_eligible_proxy_rate":float(df.loc[normal_ctr,"target"].mean()) if normal_ctr.any() else None,
    "verdict":"DIRECTIONAL"
}
for name,payload in tests.items():
    print(name,payload)


age {'negative_median': 287.0, 'positive_median': 216.0, 'verdict': 'DIRECTIONAL'}
visibility {'high_visibility_proxy_rate': 0.5939894715799293, 'low_visibility_proxy_rate': 0.4900953778429934, 'verdict': 'MIXED'}
low_ctr_visible {'low_ctr_proxy_rate': 0.6265777321703437, 'other_eligible_proxy_rate': 0.4752650176678445, 'verdict': 'DIRECTIONAL'}


## 3. The flag-linked test

The transparent refresh baseline uses a **stale + visible** idea: pages with at least 500 impressions and at least 180 days since the recorded update deserve review. I test the proxy rate in that bucket versus the rest of the starter slice.

A positive difference would support using the signal as a review heuristic; it still would not prove staleness causes decline or that refreshing the page will recover traffic.

In [3]:
stale_visible=(df["impressions_90d"]>=500)&(df["days_since_last_update"]>=180)
bucket_n=int(stale_visible.sum())
bucket_rate=float(df.loc[stale_visible,"target"].mean()) if bucket_n else float("nan")
rest_rate=float(df.loc[~stale_visible,"target"].mean())
print("stale_visible n:",bucket_n)
print("stale_visible proxy rate:",round(bucket_rate,3) if bucket_n else "n/a")
print("rest proxy rate:",round(rest_rate,3))
print("Verdict:", "DIRECTIONAL" if bucket_n and bucket_rate>rest_rate else "MIXED")


stale_visible n: 17
stale_visible proxy rate: 0.941
rest proxy rate: 0.542
Verdict: DIRECTIONAL


## 4. What this means in practice

The audit supports using multiple signals together rather than treating any single threshold as truth. A content team should read a high score as **priority for review**, then check intent, seasonality, consolidation, tracking quality, and editorial context. The fixed rule stays valuable because its assumptions are visible and can be challenged.

In [4]:
practical={
    "single_signal_is_decision":False,
    "combine_signals_for_priority":True,
    "human_review_required":True,
    "causal_refresh_claim":False,
    "recommended_use":"rank review candidates, then inspect context"
}
print(practical)


{'single_signal_is_decision': False, 'combine_signals_for_priority': True, 'human_review_required': True, 'causal_refresh_claim': False, 'recommended_use': 'rank review candidates, then inspect context'}


## Self-check

- [x] Key distributions are inspected
- [x] Three safe signals have descriptive tests and explicit verdicts
- [x] A real flag-linked stale/visible assumption is checked
- [x] Interpretation is review-oriented and non-causal
- [ ] Notebook execution outputs verified